In [1]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import ZenodoSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

/home/gpudual/bhklab/declan/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# PROMISE12

.mhd files so we are skipping.

In [2]:
DATASET_NAME = "PROMISE12"

indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])



14:50:24 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:65


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Abdomen Atlas 3',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung'
]

In [3]:
description = """
This repository contains all data associated with the 'Prostate MR Image Segmentation'-challenge 2012 on https://promise12.grand-challenge.org/. The goal of this challenge was to compare interactive and (semi)-automatic segmentation algorithms for MRI of the prostate. 
"""

In [4]:
source = ZenodoSource(
    file_type="nifti",
    record_id="8026660",
    description=description,
    post_download=["unzip"]
)

In [5]:
dataset_index_path = (indexed_datasets.path / DATASET_NAME)
print(dataset_index_path)
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

indexed_datasets/PROMISE12

In [6]:

print(indexed_datasets.source_config(DATASET_NAME))

ZenodoSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    post_download=['unzip'],
    source='zenodo',
    record_id='8026660',
    filenames=None,
    description="\nThis repository contains all data associated with the 'Prostate MR Image Segmentation'-challenge
2012 on https://promise12.grand-challenge.org/. The goal of this challenge was to compare interactive and 
(semi)-automatic segmentation algorithms for MRI of the prostate. \n"
)

In [7]:
temp_data_path = Path("temp_data") / DATASET_NAME
temp_data_path.mkdir(parents=True, exist_ok=True)
print(temp_data_path.absolute())
downloader = indexed_datasets.downloader(DATASET_NAME).download(temp_data_path)

/home/gpudual/bhklab/declan/med-image_index/temp_data/PROMISE12

14:50:25 [info     ] Downloading all files from Zenodo record 8026660 [imgnet] call=downloaders.download:123
training_data.zip: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 324M/324M [02:27<00:00, 2.20MB/s]
test_data.zip: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 195M/195M [02:36<00:00, 1.24MB/s]
livechallenge_test_data.zip: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 120M/120M [01:31<00:00, 1.30MB/s]
LICENSE.TXT: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 129/129 [00:00<00:00, 55.2kB/s]


In [8]:
import zipfile
from glob import glob
import os

def extract_zip(zip_path, extract_to=None):
    """
    Extract a zip file to a specified directory
    
    Args:
        zip_path: Path to the zip file
        extract_to: Directory to extract to (default: same directory as zip)
    """
    if extract_to is None:
        extract_to = os.path.dirname(zip_path)
    
    # Create extraction directory if it doesn't exist
    os.makedirs(extract_to, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        print(f"Extracted {zip_path} to {extract_to}")

for zip_file in glob(f"{temp_data_path}/*.zip"):
    print(zip_file)
    # Get the base name without extension to create a folder with the same name
    zip_name = os.path.splitext(os.path.basename(zip_file))[0]
    extract_to = os.path.join(f"{temp_data_path}", zip_name)
    extract_zip(zip_file, temp_data_path)

temp_data/PROMISE12/training_data.zip

Extracted temp_data/PROMISE12/training_data.zip to temp_data/PROMISE12

temp_data/PROMISE12/test_data.zip

Extracted temp_data/PROMISE12/test_data.zip to temp_data/PROMISE12

temp_data/PROMISE12/livechallenge_test_data.zip

Extracted temp_data/PROMISE12/livechallenge_test_data.zip to temp_data/PROMISE12

In [9]:

crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="final-formatted/images/{BodyPartExamined}/{training_phase}/images/{image_number}_{SeriesInstanceUID}.nii.gz",
    mask_name_pattern="final-formatted/images/{BodyPartExamined}/{training_phase}/masks/{image_number}_{SeriesInstanceUID}.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=True,
)
crawler.crawl()




FileNotFoundError: No matching files in /home/gpudual/bhklab/declan/med-image_index/temp_data/PROMISE12

In [ ]:

index_df = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")
index_df



,BodyPartExamined,training_phase,image_number,SeriesInstanceUID,filepath,file_type,reference_id,class,hash,size,...,mask.bbox.max_coord,mask.feret_diameter,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan
0,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_003_1.3.12.2.1107.5.1.4.83504.300...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021082509324422...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_003_1.3.12.2.1107.5.1.4.83504.300...,MedImage,0b0f2d0fbb1d47cc16dd4c89cb3474202b01e3ed,"(512, 512, 175)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021102712332815...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_003_1.3.12.2.1107.5.1.4.83504.300...,MedImage,bd06c947ae6e5249a8489a9ce8f37bd7e8dd8599,"(512, 512, 177)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,abdomen,test,13,1.3.12.2.1107.5.1.4.83504.30000021061014315422...,final-formatted/images/abdomen/test/images/013...,scan,abdomen_test_013_1.3.12.2.1107.5.1.4.83504.300...,MedImage,5e67e7fe10e0c53b4f105ad9d4a7fc50ad77a47a,"(512, 512, 203)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,abdomen,test,13,1.3.12.2.1107.5.1.4.83504.30000021081905213109...,final-formatted/images/abdomen/test/images/013...,scan,abdomen_test_013_1.3.12.2.1107.5.1.4.83504.300...,MedImage,3892d6a599c4f524c17779725e045e3a82b8f13d,"(512, 512, 189)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,thorax,train,22,1.3.12.2.1107.5.1.4.83504.30000021011202081810...,final-formatted/images/thorax/train/masks/022_...,mask,thorax_train_022_1.3.12.2.1107.5.1.4.83504.300...,Mask,28b8d7f41f1bf84be82243f0dd224389f10a639c,"(512, 512, 339)",...,"(178, 321, 260)",196.547417,0.314212,2.673468,2.999829,6.377344,511.080815,"(4.591145025879188, 12.27427905085932, 36.8207...",4.0,final-formatted/images/thorax/train/images/022...
112,thorax,train,22,1.3.12.2.1107.5.1.4.83504.30000021031714114711...,final-formatted/images/thorax/train/masks/022_...,mask,thorax_train_022_1.3.12.2.1107.5.1.4.83504.300...,Mask,b4bc4f8808b690dccbead0a9eb1586efb70bd83a,"(512, 512, 317)",...,"(358, 152, 162)",133.361374,0.249407,6.061300,2.719413,6.412558,516.740526,"(2.7639216438141267, 16.75295962055886, 45.558...",4.0,final-formatted/images/thorax/train/images/022...
113,thorax,train,31,1.3.12.2.1107.5.1.4.83504.30000023081013180867...,final-formatted/images/thorax/train/masks/031_...,mask,thorax_train_031_1.3.12.2.1107.5.1.4.83504.300...,Mask,ae3e4591274bc978da691359e12dfadd5db9fd50,"(512, 512, 315)",...,"(465, 261, 179)",140.242998,0.303226,1.247457,3.115993,15.849530,3156.767655,"(18.728245934568143, 23.362684662898715, 72.79...",9.0,final-formatted/images/thorax/train/images/031...
114,thorax,train,31,1.3.12.2.1107.5.1.4.83504.30000023101713284842...,final-formatted/images/thorax/train/masks/031_...,mask,thorax_train_031_1.3.12.2.1107.5.1.4.83504.300...,Mask,6161cd6a28d7001596c7009991844e204c6c5906,"(512, 512, 295)",...,"(489, 280, 145)",154.172329,0.348818,1.948031,5.355082,13.930932,2438.766530,"(10.209963199857011, 19.88932845748825, 106.50...",3.0,final-formatted/images/thorax/train/images/031...


In [ ]:
def map_modality(row):
    if row["file_type"] == "scan":
            return "MR"
    else:
        return "SEG"
    
index_df["Modality"] = index_df.apply(map_modality, axis=1)




In [ ]:
index_df

,BodyPartExamined,training_phase,image_number,SeriesInstanceUID,filepath,file_type,reference_id,class,hash,size,...,mask.feret_diameter,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality
0,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_003_1.3.12.2.1107.5.1.4.83504.300...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR
1,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021082509324422...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_003_1.3.12.2.1107.5.1.4.83504.300...,MedImage,0b0f2d0fbb1d47cc16dd4c89cb3474202b01e3ed,"(512, 512, 175)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR
2,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021102712332815...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_003_1.3.12.2.1107.5.1.4.83504.300...,MedImage,bd06c947ae6e5249a8489a9ce8f37bd7e8dd8599,"(512, 512, 177)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR
3,abdomen,test,13,1.3.12.2.1107.5.1.4.83504.30000021061014315422...,final-formatted/images/abdomen/test/images/013...,scan,abdomen_test_013_1.3.12.2.1107.5.1.4.83504.300...,MedImage,5e67e7fe10e0c53b4f105ad9d4a7fc50ad77a47a,"(512, 512, 203)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR
4,abdomen,test,13,1.3.12.2.1107.5.1.4.83504.30000021081905213109...,final-formatted/images/abdomen/test/images/013...,scan,abdomen_test_013_1.3.12.2.1107.5.1.4.83504.300...,MedImage,3892d6a599c4f524c17779725e045e3a82b8f13d,"(512, 512, 189)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,thorax,train,22,1.3.12.2.1107.5.1.4.83504.30000021011202081810...,final-formatted/images/thorax/train/masks/022_...,mask,thorax_train_022_1.3.12.2.1107.5.1.4.83504.300...,Mask,28b8d7f41f1bf84be82243f0dd224389f10a639c,"(512, 512, 339)",...,196.547417,0.314212,2.673468,2.999829,6.377344,511.080815,"(4.591145025879188, 12.27427905085932, 36.8207...",4.0,final-formatted/images/thorax/train/images/022...,SEG
112,thorax,train,22,1.3.12.2.1107.5.1.4.83504.30000021031714114711...,final-formatted/images/thorax/train/masks/022_...,mask,thorax_train_022_1.3.12.2.1107.5.1.4.83504.300...,Mask,b4bc4f8808b690dccbead0a9eb1586efb70bd83a,"(512, 512, 317)",...,133.361374,0.249407,6.061300,2.719413,6.412558,516.740526,"(2.7639216438141267, 16.75295962055886, 45.558...",4.0,final-formatted/images/thorax/train/images/022...,SEG
113,thorax,train,31,1.3.12.2.1107.5.1.4.83504.30000023081013180867...,final-formatted/images/thorax/train/masks/031_...,mask,thorax_train_031_1.3.12.2.1107.5.1.4.83504.300...,Mask,ae3e4591274bc978da691359e12dfadd5db9fd50,"(512, 512, 315)",...,140.242998,0.303226,1.247457,3.115993,15.849530,3156.767655,"(18.728245934568143, 23.362684662898715, 72.79...",9.0,final-formatted/images/thorax/train/images/031...,SEG
114,thorax,train,31,1.3.12.2.1107.5.1.4.83504.30000023101713284842...,final-formatted/images/thorax/train/masks/031_...,mask,thorax_train_031_1.3.12.2.1107.5.1.4.83504.300...,Mask,6161cd6a28d7001596c7009991844e204c6c5906,"(512, 512, 295)",...,154.172329,0.348818,1.948031,5.355082,13.930932,2438.766530,"(10.209963199857011, 19.88932845748825, 106.50...",3.0,final-formatted/images/thorax/train/images/031...,SEG


In [ ]:
index_df["filename"] = index_df['filepath'].apply(lambda x: os.path.basename(x))

clinical_df = pd.read_csv(temp_data_path / "final-formatted/recist_measurements.csv")

full_df = pd.merge(index_df, clinical_df, how="left", left_on="filename", right_on="filename")

In [ ]:
full_df

,BodyPartExamined,training_phase,image_number,SeriesInstanceUID,filepath,file_type,reference_id,class,hash,size,...,subset,study_date,study_uuid,uuid,region,final_3d_objects,lesion_label_value,lesion_label_alias,recist_measurement_mm,study_order
0,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_003_1.3.12.2.1107.5.1.4.83504.300...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,test,20210615.0,1.3.51.0.1.1.172.19.3.128.2857496.2857435,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,abdomen,3.0,3.0,A,61.0,baseline
1,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_003_1.3.12.2.1107.5.1.4.83504.300...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,test,20210615.0,1.3.51.0.1.1.172.19.3.128.2857496.2857435,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,abdomen,3.0,2.0,B,23.0,baseline
2,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_003_1.3.12.2.1107.5.1.4.83504.300...,MedImage,81ef1a4ffb1dc7d6dd92451bb3e70c5e30b7378c,"(512, 512, 173)",...,test,20210615.0,1.3.51.0.1.1.172.19.3.128.2857496.2857435,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,abdomen,3.0,1.0,C,17.0,baseline
3,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021082509324422...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_003_1.3.12.2.1107.5.1.4.83504.300...,MedImage,0b0f2d0fbb1d47cc16dd4c89cb3474202b01e3ed,"(512, 512, 175)",...,test,20210825.0,1.3.51.0.1.1.172.19.3.128.2890383.2890322,1.3.12.2.1107.5.1.4.83504.30000021082509324422...,abdomen,3.0,3.0,A,61.0,follow-up-1
4,abdomen,test,3,1.3.12.2.1107.5.1.4.83504.30000021082509324422...,final-formatted/images/abdomen/test/images/003...,scan,abdomen_test_003_1.3.12.2.1107.5.1.4.83504.300...,MedImage,0b0f2d0fbb1d47cc16dd4c89cb3474202b01e3ed,"(512, 512, 175)",...,test,20210825.0,1.3.51.0.1.1.172.19.3.128.2890383.2890322,1.3.12.2.1107.5.1.4.83504.30000021082509324422...,abdomen,3.0,2.0,B,17.0,follow-up-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189,thorax,train,22,1.3.12.2.1107.5.1.4.83504.30000021011202081810...,final-formatted/images/thorax/train/masks/022_...,mask,thorax_train_022_1.3.12.2.1107.5.1.4.83504.300...,Mask,28b8d7f41f1bf84be82243f0dd224389f10a639c,"(512, 512, 339)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190,thorax,train,22,1.3.12.2.1107.5.1.4.83504.30000021031714114711...,final-formatted/images/thorax/train/masks/022_...,mask,thorax_train_022_1.3.12.2.1107.5.1.4.83504.300...,Mask,b4bc4f8808b690dccbead0a9eb1586efb70bd83a,"(512, 512, 317)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
191,thorax,train,31,1.3.12.2.1107.5.1.4.83504.30000023081013180867...,final-formatted/images/thorax/train/masks/031_...,mask,thorax_train_031_1.3.12.2.1107.5.1.4.83504.300...,Mask,ae3e4591274bc978da691359e12dfadd5db9fd50,"(512, 512, 315)",...,train,20230810.0,1.3.51.0.1.1.172.19.3.128.3238264.3238203,1.3.12.2.1107.5.1.4.83504.30000023081013180867...,thorax,7.0,8.0,A,28.0,baseline
192,thorax,train,31,1.3.12.2.1107.5.1.4.83504.30000023101713284842...,final-formatted/images/thorax/train/masks/031_...,mask,thorax_train_031_1.3.12.2.1107.5.1.4.83504.300...,Mask,6161cd6a28d7001596c7009991844e204c6c5906,"(512, 512, 295)",...,train,20231017.0,1.3.51.0.1.1.172.19.3.128.3268322.3268261,1.3.12.2.1107.5.1.4.83504.30000023101713284842...,thorax,3.0,10.0,A,23.0,follow-up-1


In [ ]:
# from imgnet.collections.store import IndexedDatasets
# from imgindex.model import validate_index
# from imgindex.parquet import csv_to_parquet

# index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
# index_df["SampleID"] = index_df["SampleID"].astype(str)
# index_df.to_csv(indexed_datasets.path / DATASET_NAME / "index.csv", index=False)
# csv_to_parquet(index_df, indexed_datasets.path / DATASET_NAME)
# validate_index(index_df, "nifti", lazy=True)

,patient_number,scan_time,name,filepath,file_type,SampleID,class,hash,size,ndim,...,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,is_registered,Modality,ROINames,BodyPartExamined
0,10,midRT,10_midRT,HNTSMRG24_train/10/midRT/10_midRT_T2.nii.gz,scan,10_midRT_10_midRT,MedImage,e8e14d254c581cd759fef50b0d54b126c24da149,"(512, 512, 76)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,0,MR,,OROPHARYNX
1,10,midRT,10_midRT,HNTSMRG24_train/10/midRT/10_midRT_mask.nii.gz,mask,10_midRT_10_midRT,Mask,2bdaf929ea9b48c200053952ea4a0f1875146d7e,"(512, 512, 76)",3,...,1.185530,14.438213,2619.610821,"(17.639697402822986, 33.93230573452438, 40.227...",2.0,HNTSMRG24_train/10/midRT/10_midRT_T2.nii.gz,0,SEG,"GTVp,GTVn",OROPHARYNX
2,10,preRT,10_preRT,HNTSMRG24_train/10/preRT/10_preRT_T2.nii.gz,scan,10_preRT_10_preRT,MedImage,83d8db42388c09b9b41c3f980ca87290a2668778,"(512, 512, 75)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,0,MR,,OROPHARYNX
3,10,preRT,10_preRT,HNTSMRG24_train/10/preRT/10_preRT_mask.nii.gz,mask,10_preRT_10_preRT,Mask,65566f1960f7b4bf71264b3442412063e151f6e8,"(512, 512, 75)",3,...,1.676271,14.746881,2732.815072,"(15.27453786193793, 31.654766112668213, 53.061...",2.0,HNTSMRG24_train/10/preRT/10_preRT_T2.nii.gz,0,SEG,"GTVp,GTVn",OROPHARYNX
4,101,midRT,101_midRT,HNTSMRG24_train/101/midRT/101_midRT_T2.nii.gz,scan,101_midRT_101_midRT,MedImage,afcd8a4d1bdc913d3c22fe232c8935762a639b77,"(512, 512, 73)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,0,MR,,OROPHARYNX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,95,midRT,95_preRT,HNTSMRG24_train/95/midRT/95_preRT_mask_registe...,mask,95_midRT_95_preRT_registered,Mask,ad44969cff59873b27926bed9c4b114d5b89c538,"(768, 768, 116)",3,...,1.772212,17.839432,3999.188850,"(24.385387992901403, 32.41856660529757, 57.452...",3.0,HNTSMRG24_train/95/midRT/95_preRT_T2_registere...,1,SEG,"GTVp,GTVn",OROPHARYNX
896,96,midRT,96_preRT,HNTSMRG24_train/96/midRT/96_preRT_T2_registere...,scan,96_midRT_96_preRT_registered,MedImage,4edcae7ef36c696019cddeec0e24bd6e507a8781,"(512, 512, 75)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,1,MR,,OROPHARYNX
897,96,midRT,96_preRT,HNTSMRG24_train/96/midRT/96_preRT_mask_registe...,mask,96_midRT_96_preRT_registered,Mask,3c4c62c1cd41c16956fd2af3aeed79e41aae0e01,"(512, 512, 75)",3,...,3.710939,8.743070,960.589361,"(7.704017074688655, 13.6754109042931, 50.74861...",2.0,HNTSMRG24_train/96/midRT/96_preRT_T2_registere...,1,SEG,"GTVp,GTVn",OROPHARYNX
898,99,midRT,99_preRT,HNTSMRG24_train/99/midRT/99_preRT_T2_registere...,scan,99_midRT_99_preRT_registered,MedImage,fc7cba8c4b17b047b6d86c5d11b3913f0d119f89,"(512, 512, 81)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,1,MR,,OROPHARYNX
